# Notebook 03: Demand by warehouse and the first product family

**Project:** Perishable Inventory Optimizer (GIS + supply chain)
**Milestone:** 3, Demand aggregation and forecast

## What we are trying to accomplish
Turn the store-level sales into a daily history for each warehouse region, check how clean and steady each perishable family is, and choose the one family the first app will recommend stock for. Later steps build a forecast for that family and test it on weeks we hold out ourselves.

## What we got accomplished
- **Step 1 is done.** We loaded the perishable sales, attached each store's warehouse, and added the stores up into one row per day, warehouse and family (60,624 rows, none missing).
- **Step 2 is written and ready to run.** It checks which stores were open, how often each family sells nothing, and which days look unusual.

## What we have left to do
1. Step 2: run the checks and read the results together (open stores, zero-sales days, unusual days).
2. Step 3: choose the single family for the first app, and record why.
3. Then: hold out the last weeks as our own test set, build a baseline forecast and a better model, and compare them.

**How each step works:** we run the code, look at the result, and fill in the explanation cells together: what we are looking for, how to read it, what it means, and what is next and why.

**Reminders from earlier milestones:** a sale is not the same as demand, because a sold-out shelf hides unmet demand. Sales are per product family, not per item. We never use the contest files (`test.csv`, `sample_submission.csv`); our test set is the last weeks of `train.csv`. Perishable families are measured in different units (produce by weight, the others by item), so we compare each family with itself over time, not with the others.

## Step 1. Sales by warehouse, day and family

### What we are looking for
One row for every day, warehouse and perishable family, holding the units sold by all of that warehouse's stores. The forecast and the stocking recommendation work per warehouse, so this table is what they will use.

### How we got here (the process)
1. **Load the sales** from `train.csv` (all 3,000,888 rows, 2013-01-01 to 2017-08-15) and keep the nine perishable families.
2. **Add each store's warehouse** from `data/reference/store_warehouse.csv`, the file Milestone 2 produced. The code stops if any row has no warehouse.
3. **Add up the stores** in each warehouse for each day and family. We also keep the number of stores and the count of items on promotion, because a warehouse with more stores sells more and promotions can lift sales.
4. **Check the shape:** the row count must equal days x warehouses x families, so no combination is missing.
5. **Save** the result in `data/processed/perishable_daily_by_warehouse.csv`. It is not committed to GitHub (rebuild it by running this notebook).

**Assumptions and limits:** the warehouse of each store comes from our assumed sites. The store count is the number of stores assigned to the warehouse, not the number that were open on that day, so the early days can include stores that had not opened yet. Step 2 checks for this.

In [11]:
from pathlib import Path
import pandas as pd

RAW = Path("../data/raw")
REF = Path("../data/reference")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

# The nine perishable families agreed in Milestone 1.
PERISHABLE = ["PRODUCE", "DAIRY", "MEATS", "POULTRY", "SEAFOOD", "EGGS", "BREAD/BAKERY", "DELI", "PREPARED FOODS"]

sales_all = pd.read_csv(next(RAW.rglob("train.csv")), parse_dates=["date"])
print(f"All sales rows: {len(sales_all):,}  |  {sales_all['date'].min().date()} to {sales_all['date'].max().date()}")

sales = sales_all[sales_all["family"].isin(PERISHABLE)].copy()
missing = set(PERISHABLE) - set(sales["family"].unique())
assert not missing, f"Families not found in the data: {missing}"
print(f"Perishable rows: {len(sales):,} ({len(sales) / len(sales_all):.0%} of all rows)")
print(f"Stores: {sales['store_nbr'].nunique()}  |  Families: {sales['family'].nunique()}  |  Days: {sales['date'].nunique():,}")
display(sales.head())

All sales rows: 3,000,888  |  2013-01-01 to 2017-08-15
Perishable rows: 818,424 (27% of all rows)
Stores: 54  |  Families: 9  |  Days: 1,684


,id,date,store_nbr,family,sales,onpromotion
5,5,2013-01-01,1,BREAD/BAKERY,0.0,0
8,8,2013-01-01,1,DAIRY,0.0,0
9,9,2013-01-01,1,DELI,0.0,0
10,10,2013-01-01,1,EGGS,0.0,0
24,24,2013-01-01,1,MEATS,0.0,0


### How to read it
The first line counts every row in `train.csv`: 3,000,888 rows, one per store, day and family, from 2013-01-01 to 2017-08-15. The second line shows how many of those rows are perishable: 818,424, which is 27%. That matches what we expect, because 9 of the 33 families are perishable (9 / 33 = 27%). The third line says all 54 stores, all 9 families and 1,684 days are present. The small table is the first five perishable rows.

### What it means
The data loaded completely and the filter kept exactly the nine families we agreed on in Milestone 1 (the code would have stopped if any were missing). Look at the table: every sale on 2013-01-01 is 0. That is not a quiet day for every product. Stores in Ecuador are closed on New Year's Day, so zero here can mean "closed", not "nobody wanted it". We will deal with that in Step 2.

### What's next and why
The sales rows are per store. Our stocking recommendation is per warehouse, so next we attach each store's warehouse from the Milestone 2 file.

In [12]:
# 1b. Attach each store's warehouse (from Milestone 2).
store_warehouse = pd.read_csv(REF / "store_warehouse.csv")[["store_nbr", "city", "warehouse"]]
sales = sales.merge(store_warehouse[["store_nbr", "warehouse"]], on="store_nbr", how="left", validate="many_to_one")
assert sales["warehouse"].notna().all(), "Some sales rows have no warehouse"

rows_by_wh = sales.groupby("warehouse").agg(
    stores=("store_nbr", "nunique"), rows=("sales", "size"), total_units=("sales", "sum")
)
rows_by_wh["share_of_units"] = (rows_by_wh["total_units"] / rows_by_wh["total_units"].sum()).round(3)
display(rows_by_wh.sort_values("total_units", ascending=False))

,stores,rows,total_units,share_of_units
warehouse,,,,
Quito,25,378900,2.212372e+08,0.645
Guayaquil,16,242496,6.946700e+07,0.203
Cuenca,7,106092,3.330198e+07,0.097
Santo Domingo,6,90936,1.879658e+07,0.055


### How to read it
Each row is a warehouse. `stores` is how many stores it serves, `rows` is how many sales rows belong to it (stores x 1,684 days x 9 families), `total_units` is everything those stores sold over the whole history, and `share_of_units` is that warehouse's share of the total.

| Warehouse | Stores | Share of units |
|---|---|---|
| Quito | 25 | 64.5% |
| Guayaquil | 16 | 20.3% |
| Cuenca | 7 | 9.7% |
| Santo Domingo | 6 | 5.5% |

### What it means
Every store found a warehouse (the code stops if one did not, and the many-to-one check would have caught duplicates). Quito carries almost two thirds of all perishable units, so its forecast and its recommendations matter most. Notice that Quito has 46% of the stores (25 of 54) but 64.5% of the units, so its stores sell more each than the others on average. Santo Domingo is the smallest, with 6 stores and 5.5% of units, so its daily numbers will be small and noisier. These units mix produce (weight) and the other families (items), so the totals show size, not a clean count of products.

### What's next and why
Now we add the stores together for each day, warehouse and family. That gives the history the forecast will learn from.

In [13]:
# 1c. Add up the stores in each warehouse, by day and family.
# The store count matters later: a warehouse with more stores sells more, so we keep it beside the totals.
daily = (
    sales.groupby(["date", "warehouse", "family"], as_index=False)
    .agg(sales=("sales", "sum"), stores=("store_nbr", "nunique"), on_promotion=("onpromotion", "sum"))
)
DAILY_FILE = PROCESSED / "perishable_daily_by_warehouse.csv"
daily.to_csv(DAILY_FILE, index=False)

expected_rows = daily["date"].nunique() * daily["warehouse"].nunique() * daily["family"].nunique()
print(f"{len(daily):,} rows (days x warehouses x families = {expected_rows:,})")
print(f"Saved to {DAILY_FILE}")
display(daily.head(8))

60,624 rows (days x warehouses x families = 60,624)
Saved to ..\data\processed\perishable_daily_by_warehouse.csv


,date,warehouse,family,sales,stores,on_promotion
0,2013-01-01,Cuenca,BREAD/BAKERY,0.0,7,0
1,2013-01-01,Cuenca,DAIRY,0.0,7,0
2,2013-01-01,Cuenca,DELI,0.0,7,0
3,2013-01-01,Cuenca,EGGS,0.0,7,0
4,2013-01-01,Cuenca,MEATS,0.0,7,0
5,2013-01-01,Cuenca,POULTRY,0.0,7,0
6,2013-01-01,Cuenca,PREPARED FOODS,0.0,7,0
7,2013-01-01,Cuenca,PRODUCE,0.0,7,0


### How to read it
The first line compares the rows we made, 60,624, with days x warehouses x families (1,684 x 4 x 9), which is also 60,624. They match, so no combination is missing and none is repeated. The table shows the first rows: one per family for Cuenca on 2013-01-01. `sales` is the units sold by all the warehouse's stores, `stores` is the number of stores assigned to it (7 for Cuenca), and `on_promotion` is how many items were on promotion.

### What it means
We now have one tidy table, saved to `data/processed/perishable_daily_by_warehouse.csv` (not on GitHub; rerun this notebook to rebuild it). All the early rows show 0 sales for all seven Cuenca stores, which again points to New Year's Day closures. Also, `stores` is the number of stores assigned to the warehouse, not the number open that day, so on days when some stores were closed or not yet opened, this table overstates how many stores were selling. That is the limit noted at the top of this step, and Step 2 measures it.

### What's next and why
Before choosing a family or forecasting anything, we need to know which zeros are real "no sales" days and which are closed or not-yet-open stores. Step 2 separates them.

### Step 1 conclusion
- **What we did:** loaded 3,000,888 sales rows, kept the 818,424 perishable ones, attached each store's warehouse, and added the stores up by day, warehouse and family.
- **What we found:** Quito sells 64.5% of the perishable units, Guayaquil 20.3%, Cuenca 9.7%, Santo Domingo 5.5%. The table has exactly the 60,624 rows we expected.
- **What it means for the project:** we have a clean, complete table for the forecast, but it still counts closed and not-yet-open stores as selling zero.
- **What's next:** Step 2 separates real zero-sales days from closed stores and finds unusual days.

## Step 2. Closed stores, zero-sales days and unusual days

### What we are looking for
Zeros and spikes in the sales that are not real demand. A forecast that learns from a day when the store was closed will think demand fell to nothing, and a stock recommendation built on it would be wrong. We want to know:
1. Which stores had not opened yet at the start of the data, and which days stores were closed.
2. How often each family sells nothing on days the store was open, which tells us how reliable each family's history is.
3. Which warehouse days look unusually high or low compared with the days around them.

### How we got here (the process)
We can only see sales, not opening hours, so we infer them.
1. **Open or closed (2a).** If a store sold nothing in any of the 33 families on a day, it was not open. Before its first-ever sale we call it *not yet open*; after that, a day of total zero is a *closure*. We chose this over looking only at perishable families, because a family can legitimately sell zero while the store is open, and the all-family total cannot be zero for an open store. We checked it with the dates when the most stores were closed (we expect New Year's Day).
2. **Zero share per family (2b).** For each perishable family we compare the share of zero rows across all rows with the share on open days only. If the two differ a lot, the zeros came from closures. If a family still has many zeros on open days, its history may be unreliable or it may truly sell rarely.
3. **Zeros by year (2c).** For the families that still have zeros, we look at each year, since a family could have been recorded differently in the early years.
4. **Unusual days (2d).** Using open stores only, we compute sales per open store, compare each day with the median of the surrounding 29 days for the same warehouse and family, and mark days more than 3 times above or below that level. We considered a fixed cut-off (such as the 99th percentile) and rejected it, because families and warehouses have very different sizes, and a fixed number would flag only the biggest warehouse. The ratio to the local typical level works for every warehouse and family. The cut-off of 3 is a judgement call; we can test other values after seeing the results.
5. **Save** the checked table to `data/processed/perishable_daily_open.csv` (not on GitHub).

**Assumptions and limits:** a day with zero sales in every family is treated as closed, which could also be a missing-data day. We cannot tell the two apart. Flagged days are only *marked*, not removed, until we decide together what to do with them.

In [14]:
# 2a. Which stores were open on which days?
# A store-day with no sales in ANY family means the store was not open that day.
# Before a store's first sale it had not opened; after it, a zero day is a closure.
import numpy as np

store_day = (
    sales_all.groupby(["store_nbr", "date"], as_index=False)["sales"].sum().rename(columns={"sales": "store_total"})
)
first_sale = store_day[store_day["store_total"] > 0].groupby("store_nbr")["date"].min().rename("first_sale").reset_index()
store_day = store_day.merge(first_sale, on="store_nbr")
store_day["not_yet_open"] = store_day["date"] < store_day["first_sale"]
store_day["closed"] = (~store_day["not_yet_open"]) & (store_day["store_total"] == 0)
store_day["is_open"] = ~(store_day["not_yet_open"] | store_day["closed"])

first_day = sales_all["date"].min()
late = (
    store_warehouse[["store_nbr", "city", "warehouse"]].merge(first_sale, on="store_nbr")
    .assign(days_after_start=lambda d: (d["first_sale"] - first_day).dt.days)
    .query("days_after_start > 1")
    .sort_values("first_sale")
)
print(f"Store-days in the data: {len(store_day):,}")
print(f"  not yet open: {store_day['not_yet_open'].sum():,}   closed after opening: {store_day['closed'].sum():,}   open: {store_day['is_open'].sum():,}")
print(f"\nStores that opened after the data begins: {len(late)} of {store_day['store_nbr'].nunique()}")
display(late)

closed_by_date = store_day[store_day["closed"]].groupby("date").size().sort_values(ascending=False).head(6)
closed_by_store = store_day[store_day["closed"]].groupby("store_nbr").size().sort_values(ascending=False).head(6)
print("\nDates when the most stores were closed:")
display(closed_by_date.rename("stores_closed").to_frame())
print("Stores with the most closed days:")
display(closed_by_store.rename("closed_days").to_frame())

Store-days in the data: 90,936
  not yet open: 6,729   closed after opening: 601   open: 83,606

Stores that opened after the data begins: 8 of 54


,store_nbr,city,warehouse,first_sale,days_after_start
35,36,Libertad,Guayaquil,2013-05-09,128
52,53,Manta,Guayaquil,2014-05-29,513
19,20,Quito,Quito,2015-02-13,773
28,29,Guayaquil,Guayaquil,2015-03-20,808
20,21,Santo Domingo,Santo Domingo,2015-07-24,934
41,42,Cuenca,Cuenca,2015-08-21,962
21,22,Puyo,Quito,2015-10-09,1011
51,52,Manta,Guayaquil,2017-04-20,1570



Dates when the most stores were closed:


,stores_closed
date,
2017-01-01,52
2016-01-01,52
2015-01-01,47
2014-01-01,45
2016-11-15,2
2016-10-17,2


Stores with the most closed days:


,closed_days
store_nbr,
18,114
24,105
25,66
12,64
14,42
30,27


### How to read it
- **First block:** there are 90,936 store-days (54 stores x 1,684 days). Of those, 6,729 are *not yet open* (before the store's first sale), 601 are *closed after opening*, and 83,606 are *open*. So about 92% of store-days are open.
- **Stores that opened after the data begins:** 8 of 54. `days_after_start` is how long after 2013-01-01 the store first sold anything (for example, store 36 in Libertad started 128 days in; store 52 in Manta started 1,570 days in, on 2017-04-20, only about four months before the data ends).
- **Dates when the most stores were closed:** four New Year's Days (2014 to 2017) with 45, 47, 52 and 52 stores closed. Every other date has only 1 or 2 stores closed (the table shows the two highest, 2016-11-15 and 2016-10-17, with 2 each).
- **Stores with the most closed days:** store 18 (114 days), store 24 (105), then 25, 12, 14 and 30 with 66 down to 27.

### What it means
Closed and not-yet-open days are real and cannot be ignored. If we leave them in, the early history of those 8 stores looks like zero demand, which would drag down their warehouses' forecasts (the late Manta store and the two Quito-area stores matter most for Guayaquil and Quito). New Year's Day is the one date when almost every store is closed: 196 of the 601 closures happen on those four days. (New Year's Day 2013 is counted as "not yet open" for every store, since no store had sold anything yet.) The other ~400 closures are spread over individual stores, and the two largest (stores 18 and 24, over 100 days each) look like long stretches such as a refit or temporary closure. We cannot tell the reason from the sales alone, so we only know those days are not real demand.

### What's next and why
Now that each store-day is labelled open or not, 2b checks how often each family sells nothing, with and without the closed days. That shows how many of the zeros we saw earlier were really closures.

In [15]:
# 2b. How often does each family sell nothing?
# Compare all rows with only the rows where the store was open.
sales_open = sales.merge(store_day[["store_nbr", "date", "is_open"]], on=["store_nbr", "date"], how="left")
assert sales_open["is_open"].notna().all()

zero_share = pd.DataFrame({
    "zero_share_all_rows": sales_open.groupby("family")["sales"].apply(lambda s: (s == 0).mean()),
    "zero_share_open_days": sales_open[sales_open["is_open"]].groupby("family")["sales"].apply(lambda s: (s == 0).mean()),
    "total_units": sales_open.groupby("family")["sales"].sum(),
}).round(3).sort_values("zero_share_open_days")
display(zero_share)

,zero_share_all_rows,zero_share_open_days,total_units
family,,,
BREAD/BAKERY,0.081,0.000,4.213395e+07
DAIRY,0.081,0.000,6.448771e+07
DELI,0.081,0.000,2.411032e+07
EGGS,0.081,0.000,1.558830e+07
MEATS,0.081,0.000,3.108647e+07
POULTRY,0.081,0.000,3.187600e+07
PREPARED FOODS,0.081,0.000,8.799895e+06
SEAFOOD,0.194,0.123,2.015432e+06
PRODUCE,0.284,0.221,1.227047e+08


### How to read it
Each row is a perishable family. `zero_share_all_rows` is the share of store-day rows with zero sales, counting every day. `zero_share_open_days` is the same share counting only days the store was open (from 2a). `total_units` is everything sold in the whole history.

- **Seven families** (BREAD/BAKERY, DAIRY, DELI, EGGS, MEATS, POULTRY, PREPARED FOODS): 8.1% of rows are zero across all days, and **0.0%** on open days.
- **SEAFOOD:** 19.4% zero across all days, **12.3%** on open days. It also has by far the smallest volume (about 2.0 million units, against 64 million for DAIRY).
- **PRODUCE:** 28.4% zero across all days, **22.1%** on open days. It has the second largest volume (about 123 million units).

### What it means
For the seven families, every zero was a closed or not-yet-open store. The 8.1% matches the closed and not-yet-open share from 2a (7,330 of 90,936 store-days, 8.1%). On days those stores were open, they always sold something, so a zero in these families means "closed", not "no demand". After we drop the closed days, their history is clean.

SEAFOOD and PRODUCE still have zeros on open days, which is a different problem. SEAFOOD is a small, slow seller, so a single store can honestly sell none in a day. PRODUCE is a big seller, so 22% zeros on open days is not believable as real demand. Something else is going on, and 2c looks at when.

### What's next and why
For these two families we check how the zeros change by year. If PRODUCE's zeros are concentrated in the early years, that would point to a recording problem rather than shoppers.

In [16]:
# 2c. Zero days by year for the two families that still have them.
open_rows = sales_open[sales_open["is_open"]].assign(zero=lambda d: d["sales"] == 0, year=lambda d: d["date"].dt.year)
by_year = open_rows[open_rows["family"].isin(["PRODUCE", "SEAFOOD"])].pivot_table(
    index="family", columns="year", values="zero", aggfunc="mean"
).round(3)
display(by_year)

year,2013,2014,2015,2016,2017
family,,,,,
PRODUCE,0.630,0.221,0.219,0.000,0.000
SEAFOOD,0.149,0.181,0.109,0.088,0.081


### How to read it
Each number is the share of open store-days with zero sales for that family in that year (0.630 = 63%).

| Family | 2013 | 2014 | 2015 | 2016 | 2017 |
|---|---|---|---|---|---|
| PRODUCE | 63% | 22% | 22% | 0% | 0% |
| SEAFOOD | 15% | 18% | 11% | 9% | 8% |

### What it means
**PRODUCE** zeros are a step change, not a gradual fade: 63% in 2013, about 22% in 2014 and 2015, then exactly 0% in 2016 and 2017. A real shift in shopping habits would not drop to exactly zero overnight. This looks like the way sales were recorded changed, so that before 2016 many stores' produce sales were not captured (or were recorded elsewhere). We cannot confirm the cause from this data, but we can treat the pre-2016 PRODUCE history as unreliable.

**SEAFOOD** zeros fall slowly from about 15% to 8%. That is the pattern of a small product slowly getting more popular or better stocked, not a recording change. It still has a lot of zero days, and it is small, so its forecast will be noisy.

### What's next and why
2d looks for unusual days in all nine families, using open stores only. The PRODUCE history will be hard for that check, since its early years are mostly zeros.

In [17]:
# 2d. Daily table using only open stores, and a check for unusual days.
# An unusual day is one where sales per open store are more than 3 times, or less than a third of,
# the typical level (the median of the surrounding 29 days) for that warehouse and family.
OUTLIER_RATIO = 3

open_daily = (
    sales_open[sales_open["is_open"]]
    .groupby(["date", "warehouse", "family"], as_index=False)
    .agg(sales=("sales", "sum"), open_stores=("store_nbr", "nunique"))
)
checked = daily[["date", "warehouse", "family"]].merge(open_daily, on=["date", "warehouse", "family"], how="left")
checked["open_stores"] = checked["open_stores"].fillna(0).astype(int)
checked["sales"] = checked["sales"].fillna(0)
checked["sales_per_store"] = checked["sales"] / checked["open_stores"].replace(0, np.nan)
checked = checked.sort_values(["warehouse", "family", "date"]).reset_index(drop=True)
checked["typical"] = checked.groupby(["warehouse", "family"])["sales_per_store"].transform(
    lambda s: s.rolling(29, center=True, min_periods=15).median()
)
checked["ratio"] = checked["sales_per_store"] / checked["typical"]
usable = checked["open_stores"].gt(0) & checked["typical"].gt(0)
checked["unusual"] = usable & ((checked["ratio"] > OUTLIER_RATIO) | (checked["ratio"] < 1 / OUTLIER_RATIO))

OPEN_FILE = PROCESSED / "perishable_daily_open.csv"
checked.to_csv(OPEN_FILE, index=False)
print(f"Warehouse-days with no open store (left out of the checks): {(checked['open_stores'] == 0).sum():,}")
print(f"Warehouse-days checked: {usable.sum():,}   unusual: {checked['unusual'].sum():,} ({checked['unusual'].sum() / usable.sum():.1%})")
print(f"Saved to {OPEN_FILE}")

unusual = checked[checked["unusual"]].assign(direction=lambda d: np.where(d["ratio"] > 1, "spike", "drop"))
display(unusual.pivot_table(index="family", columns="direction", values="sales", aggfunc="size", fill_value=0))
print("Largest spikes:")
display(unusual.sort_values("ratio", ascending=False).head(6)[["date", "warehouse", "family", "sales", "open_stores", "typical", "ratio"]].round({"typical": 1, "ratio": 1}))

Warehouse-days with no open store (left out of the checks): 135
Warehouse-days checked: 60,195   unusual: 587 (1.0%)
Saved to ..\data\processed\perishable_daily_open.csv


direction,drop,spike
family,,
EGGS,0,14
MEATS,0,13
POULTRY,2,2
PREPARED FOODS,1,0
PRODUCE,285,263
SEAFOOD,6,1


Largest spikes:


,date,warehouse,family,sales,open_stores,typical,ratio
8108,2016-10-07,Cuenca,MEATS,90952.064004,7,189.0,68.7
42174,2013-03-16,Quito,PRODUCE,330.000000,23,1.0,15.0
57335,2013-03-21,Santo Domingo,PRODUCE,15.000000,5,0.2,15.0
54891,2015-10-03,Santo Domingo,POULTRY,13057.347000,6,165.9,13.1
27018,2013-03-16,Guayaquil,PRODUCE,102.000000,12,0.7,12.8
42628,2014-06-14,Quito,PRODUCE,1151.000000,23,4.4,11.3


### How to read it
- **135 rows had no open store** at all (for example, a whole warehouse closed on New Year's Day). We leave them out of the check because there is no normal selling to compare with.
- **60,195 rows were checked**, and **587 (1.0%)** are unusual: more than 3 times above, or below a third of, the typical level of the surrounding 29 days. `typical` is the median sales per open store.
- **The family table** counts them: PRODUCE has 285 drops and 263 spikes. EGGS has 14 spikes, MEATS 13, SEAFOOD 7, POULTRY 4, PREPARED FOODS 1. The other families have none.
- **The largest spikes table** shows the six biggest ratios. `ratio` is that day's sales per open store divided by the typical level.

### What it means
- **PRODUCE is 548 of the 587 flags (93%).** The top produce "spikes" have a typical level of 1.0, 0.7 and 0.2 per store, which is the early-years problem from 2c. The check compared a normal day with a near-zero background, so these are not real spikes. They are more evidence that early PRODUCE is unreliable.
- **Two real outliers stand out.** MEATS in Cuenca on 2016-10-07 sold 90,952 units with 7 stores open, when the typical level was 189 per store: 69 times normal. POULTRY in Santo Domingo on 2015-10-03 sold 13,057, about 13 times normal. A one-day jump that large in a perishable product is much more likely an entry error than real demand, though the data cannot prove it.
- **The other six families are clean.** Out of tens of thousands of checked days, only a few dozen are flagged.

### What's next and why
We have enough to choose a family. The seven families with no zeros on open days are the reliable ones. PRODUCE is unreliable before 2016 and SEAFOOD is tiny and noisy. In Step 3 we pick one family from the seven. We will compare them on size, steadiness and how well the perishable-waste story fits, then decide on the two outlier days (the plan is to flag them and replace them with the typical level, not delete the rows).